In [2]:
# ============================================================
# PFE Pruning Experiments v6 — Final Publication-Grade Framework
# ResNet-50 / CIFAR-10
#
# This is the definitive version. All known issues from v3–v5 are resolved.
#
# ── EXPERIMENTAL DESIGN PRINCIPLES (enforced throughout) ─────
#
#   P1. ONE VARIABLE AT A TIME
#       Each paired comparison isolates exactly one variable:
#         1 vs 7  →  global vs per-layer threshold  (both: no fine-tune)
#         3 vs 7  →  fine-tune vs no fine-tune       (both: per-layer L1)
#         4 vs 3  →  criterion quality               (both: per-layer, same ft budget)
#         6 vs 3  →  iterative vs one-shot pruning   (both: same total ft epochs)
#
#   P2. CONTROLLED FINE-TUNE BUDGET
#       All methods that include fine-tuning use exactly FINETUNE_EPOCHS epochs
#       (default 10). Iterative pruning splits this equally across rounds so
#       total ft epochs = FINETUNE_EPOCHS for all methods.
#
#   P3. LTH TRAINING PARITY
#       LTH trains for LTH_TRAIN_EPOCHS + LTH_RETRAIN_EPOCHS. The baseline
#       was trained for BASELINE_EPOCHS. These are documented explicitly in
#       the JSON so readers can assess compute asymmetry.
#
#   P4. MASK SCOPE CONSISTENCY
#       LTH masks only Conv2d + Linear weight tensors (standard practice).
#       BN affine params are small-magnitude by normalization dynamics,
#       not by unimportance. Masking them kills feature maps silently.
#
#   P5. RELIABLE TIMING
#       All models are timed in a single dedicated pass at the end of the
#       script with torch.cuda.synchronize() barriers.
#
# ── COMPLETE FIX HISTORY ─────────────────────────────────────
#
#   From v3:
#     v3→FIX-1  Method 7 (one-shot) uses layer-wise threshold, distinct from
#               Method 1 (global). Previously both were identical global L1.
#     v3→FIX-2  Iterative make_permanent() moved outside the prune loop.
#               Masks now accumulate correctly → actual sparsity reaches 50%.
#     v3→FIX-3  Method 4 renamed "Taylor Sensitivity" (Molchanov 2017).
#               Was mislabeled "Movement Pruning" (Sanh 2020, different algo).
#
#   From v4:
#     v4→FIX-A  Method 3 (Magnitude) now includes fine-tuning.
#               Previously Methods 3 and 7 were bit-for-bit identical.
#     v4→FIX-B  LTH mask reverted to Conv2d+Linear only. v4's FIX-4
#               erroneously included BN affine → LTH accuracy dropped
#               0.9334→0.9276. Reverted.
#     v4→FIX-C  inference_ms uses torch.cuda.synchronize(). Without it,
#               CUDA async launches make wall-clock timing meaningless.
#
#   From v5 (new in v6):
#     v5→FIX-I  Method 4 (Taylor Sensitivity) now includes fine-tuning
#               matching Method 3's budget. Previously Method 4 had no
#               fine-tuning while Method 3 did, conflating criterion quality
#               with recovery strategy in any accuracy comparison.
#     v5→FIX-II Fine-tune budget equalized: all methods that fine-tune
#               use exactly FINETUNE_EPOCHS=10. Iterative splits across
#               rounds so total = FINETUNE_EPOCHS (not 15 vs 10).
#     v5→FIX-III LTH training budget documented explicitly relative to
#               baseline. BASELINE_EPOCHS config added.
#     v5→FIX-IV _is_conv_linear_weight() now includes a count assertion
#               to catch silent failures on non-standard module layouts.
#     v5→FIX-V  Method 1 (global unstructured + fine-tune variant) added
#               as Method 1b so that global vs per-layer threshold
#               comparison is not confounded by fine-tuning presence.
#               Method 1a = global, no ft. Method 1b = global + ft.
#               This makes the 1a/7 and 1b/3 comparisons both clean.
#
# ── METHOD TAXONOMY ──────────────────────────────────────────
#
#   1a_unstructured_noft   mask  global L1,    no ft      ┐ isolates threshold
#   7_one_shot             mask  per-layer L1, no ft      ┘ strategy
#
#   7_one_shot             mask  per-layer L1, no ft      ┐ isolates fine-tune
#   3_magnitude            mask  per-layer L1, ft=10ep    ┘ value
#
#   3_magnitude            mask  per-layer L1,      ft=10ep  ┐ isolates
#   4_taylor_sensitivity   mask  per-layer Taylor,  ft=10ep  ┘ criterion
#
#   3_magnitude            mask  1-shot, ft=10ep           ┐ isolates
#   6_iterative            mask  3-round, ft=10ep total    ┘ schedule
#
#   1b_unstructured_ft     mask  global L1, ft=10ep        ┐ isolates
#   3_magnitude            mask  per-layer L1, ft=10ep     ┘ threshold (with ft)
#
#   2_structured           arch  channel removal, ft=10ep  (real FLOP reduction)
#   5_lottery_ticket       rewind  θ₀→θ_T→mask→retrain    (subnetwork hypothesis)
#
# Output: __3__pruning_results_v6.json
# ============================================================

In [ ]:
import torch, torchvision, time, os, json, copy, random
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
from thop import profile
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

g = torch.Generator(); g.manual_seed(SEED)

# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
BASELINE    = "../__2__baseline_resnet50_cifar10.pth"
OUTPUT_JSON = "__3__pruning_results_v6.json"
CIFAR_MEAN  = (0.4914, 0.4822, 0.4465)
CIFAR_STD   = (0.2023, 0.1994, 0.2010)
NUM_CLASSES = 10

# Pruning targets
SPARSITY            = 0.50   # weight-level sparsity for all mask methods
CHANNEL_PRUNE_RATIO = 0.30   # fraction of bottleneck channels removed (structured)

# ── CONTROLLED FINE-TUNE BUDGET (v5→FIX-II) ──────────────────
# Every method that fine-tunes uses exactly this many total epochs.
# Iterative pruning splits this across N_ROUNDS equally.
FINETUNE_EPOCHS = 10
FINETUNE_LR     = 1e-3

# Iterative pruning rounds — fine-tune epochs per round = FINETUNE_EPOCHS // N_ROUNDS
N_ROUNDS        = 3   # must divide FINETUNE_EPOCHS evenly (10 // 3 = 3, remainder handled below)
# Epochs per round: distribute FINETUNE_EPOCHS as evenly as possible
_base_ft        = FINETUNE_EPOCHS // N_ROUNDS           # 3
_extra_ft       = FINETUNE_EPOCHS - _base_ft * N_ROUNDS # 1 (given to round 1)
ITER_FT_EPOCHS  = [_base_ft + (1 if r == 0 else 0) for r in range(N_ROUNDS)]  # [4,3,3]
assert sum(ITER_FT_EPOCHS) == FINETUNE_EPOCHS, "Iterative ft budget must sum to FINETUNE_EPOCHS"

# LTH training budget (v5→FIX-III: documented relative to baseline)
BASELINE_EPOCHS    = 50   # epochs used to train the baseline checkpoint
LTH_TRAIN_EPOCHS   = 50   # θ₀ → θ_T  (matched to baseline for fair ticket quality)
LTH_RETRAIN_EPOCHS = 50   # rewind + retrain winning ticket

# Timing config (v4→FIX-C)
TIMING_WARMUP = 50
TIMING_REPS   = 500

print(f"Device          : {DEVICE}")
print(f"Sparsity        : {SPARSITY*100:.0f}%")
print(f"Channel ratio   : {CHANNEL_PRUNE_RATIO*100:.0f}%")
print(f"Fine-tune budget: {FINETUNE_EPOCHS} epochs for all methods")
print(f"Iterative ft    : {ITER_FT_EPOCHS} epochs per round (total={sum(ITER_FT_EPOCHS)})")
print(f"LTH epochs      : {LTH_TRAIN_EPOCHS} train + {LTH_RETRAIN_EPOCHS} retrain")
print(f"Baseline epochs : {BASELINE_EPOCHS} (for LTH budget context)")

# ── MODEL FACTORY ─────────────────────────────────────────────
def build_model(num_classes=NUM_CLASSES):
    m = models.resnet50(pretrained=False)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m

def load_baseline():
    m = build_model()
    m.load_state_dict(torch.load(BASELINE, map_location=DEVICE))
    return m.to(DEVICE)

# ── DATA ──────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

train_set    = torchvision.datasets.CIFAR10('../data', True,  download=True, transform=transform_train)
test_set     = torchvision.datasets.CIFAR10('../data', False, download=True, transform=transform_test)
train_loader = torch.utils.data.DataLoader(
    train_set, BATCH_SIZE, shuffle=True,  num_workers=0,
    worker_init_fn=seed_worker, generator=g)
test_loader  = torch.utils.data.DataLoader(
    test_set,  BATCH_SIZE, shuffle=False, num_workers=0)

# ── METRIC HELPERS ────────────────────────────────────────────
def evaluate_full(model):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            preds.extend(model(x.to(DEVICE)).argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    p, l = np.array(preds), np.array(labels)
    return dict(
        accuracy  = float(accuracy_score(l, p)),
        precision = float(precision_score(l, p, average='macro', zero_division=0)),
        recall    = float(recall_score(l, p, average='macro', zero_division=0)),
        f1        = float(f1_score(l, p, average='macro', zero_division=0)),
    )

def count_params(model):  return int(sum(p.numel() for p in model.parameters()))
def count_nonzero(model): return int(sum((p != 0).sum().item() for p in model.parameters()))

def model_size_mb(model, path="_tmp_.pth"):
    torch.save(model.state_dict(), path)
    mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return float(mb)

def compute_flops(model):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return float(macs * 2 / 1e9)

def inference_ms_fn(model):
    """
    Single-sample latency with CUDA synchronization (v4→FIX-C).
    Called once per model in the dedicated end-of-script timing pass.
    """
    model.eval()
    dummy  = torch.randn(1, 3, 32, 32).to(DEVICE)
    is_gpu = DEVICE.type == "cuda"
    with torch.no_grad():
        for _ in range(TIMING_WARMUP):
            model(dummy)
        if is_gpu: torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(TIMING_REPS):
            model(dummy)
        if is_gpu: torch.cuda.synchronize()
    return float((time.perf_counter() - t0) / TIMING_REPS * 1000)

def collect_metrics(model, flops_note=None):
    """Collect all metrics except inference_ms (filled in timing pass)."""
    m   = evaluate_full(model)
    tot = count_params(model)
    nz  = count_nonzero(model)
    m.update(dict(
        params_total    = tot,
        params_nonzero  = nz,
        sparsity_actual = float(1 - nz / tot),
        size_mb         = model_size_mb(model),
        flops_G         = compute_flops(model),
        inference_ms    = None,   # filled by end-of-script timing pass
    ))
    if flops_note:
        m["flops_note"] = flops_note
    return m

FLOPS_NOTE_MASK = (
    "Mask-based pruning — computation graph unchanged. "
    "Dense FLOPs identical to baseline. "
    "Effective FLOPs require sparse-kernel backend (e.g. DeepSparse)."
)

# ── TRAINING UTILITIES ────────────────────────────────────────
def train_model(model, epochs, lr=0.1, desc="train", frozen_mask=None):
    """
    Full SGD + CosineAnnealingLR training loop.
    frozen_mask: {param_name: binary_tensor} — zeroes masked weights after
                 every optimizer step (required for correct LTH retraining).
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
            if frozen_mask is not None:
                with torch.no_grad():
                    for name, p in model.named_parameters():
                        if name in frozen_mask:
                            p.mul_(frozen_mask[name])
        scheduler.step()
        if (ep + 1) % 10 == 0 or ep == 0:
            acc = evaluate_full(model)["accuracy"]
            print(f"    [{desc}] ep {ep+1:>3}/{epochs}  acc={acc:.4f}")
    return model

def fine_tune(model, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
              desc="ft", frozen_mask=None):
    return train_model(model, epochs, lr=lr, desc=desc, frozen_mask=frozen_mask)

def get_prunable_layers(model):
    """(module, 'weight') for all Conv2d and Linear — used with torch.prune API."""
    return [(mod, 'weight') for mod in model.modules()
            if isinstance(mod, (nn.Conv2d, nn.Linear))]

def make_permanent(model):
    """Bake mask into weight tensor and remove reparametrization buffers."""
    for mod, _ in get_prunable_layers(model):
        try:  prune.remove(mod, 'weight')
        except ValueError: pass
    return model

# ── LTH MASK HELPER (v5→FIX-IV) ─────────────────────────────
def get_conv_linear_weight_keys(model):
    """
    Returns state-dict keys for Conv2d.weight and Linear.weight only.
    Excludes BN affine (γ, β), biases, and running stats.

    v5→FIX-IV: includes assertion to catch silent failures if module
    layout changes (e.g. custom modules with list attributes).
    """
    keys = []
    for name, mod in model.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            key = f"{name}.weight" if name else "weight"
            keys.append(key)
    # Sanity check: ResNet-50 CIFAR variant has 53 conv+linear weight tensors
    assert len(keys) >= 50, (
        f"_get_conv_linear_weight_keys found only {len(keys)} keys. "
        "Expected ≥50 for ResNet-50. Check model architecture."
    )
    return keys

# ── MODEL REGISTRY for end-of-script timing ──────────────────
model_registry = {}   # key → nn.Module
results        = {}   # key → metrics dict

Device          : cuda
Sparsity        : 50%
Channel ratio   : 30%
Fine-tune budget: 10 epochs for all methods
Iterative ft    : [4, 3, 3] epochs per round (total=10)
LTH epochs      : 100 train + 100 retrain
Baseline epochs : 100 (for LTH budget context)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
# ══════════════════════════════════════════════════════════════
# METHOD 1a — UNSTRUCTURED PRUNING (global L1, no fine-tune)
# ──────────────────────────────────────────────────────────────
# Global threshold: the SPARSITY-fraction of smallest-magnitude weights
# across ALL layers combined are zeroed simultaneously.
# No fine-tuning → "deploy immediately" baseline for global thresholding.
#
# Paired comparisons:
#   1a vs 7   → global vs per-layer threshold  (both: no ft)      [P1]
#   1a vs 1b  → effect of fine-tuning on global threshold          [P1]
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("1a. UNSTRUCTURED PRUNING  (global L1, no fine-tune)")
print("="*65)
model = load_baseline()
prune.global_unstructured(
    get_prunable_layers(model),
    pruning_method=prune.L1Unstructured,
    amount=SPARSITY
)
make_permanent(model)
results["1a_unstructured_noft"] = collect_metrics(model, FLOPS_NOTE_MASK)
results["1a_unstructured_noft"]["method_note"] = (
    "Global L1 unstructured pruning, no fine-tuning. "
    "Single threshold across all layers. "
    "Paired with 7_one_shot (same ft=none, different threshold scope) "
    "and 1b_unstructured_ft (same threshold, adds fine-tuning)."
)
model_registry["1a_unstructured_noft"] = model
print(f"  acc={results['1a_unstructured_noft']['accuracy']:.4f}  "
      f"sparsity={results['1a_unstructured_noft']['sparsity_actual']:.3f}")

# ══════════════════════════════════════════════════════════════
# METHOD 1b — UNSTRUCTURED PRUNING (global L1, WITH fine-tune)
# ──────────────────────────────────────────────────────────────
# Same as 1a but followed by FINETUNE_EPOCHS epochs of recovery.
# (v5→FIX-V) Added so that 1b vs 3 cleanly isolates global vs per-layer
# threshold with fine-tuning controlled.
#
# Paired comparisons:
#   1a vs 1b  → value of fine-tuning on global-pruned model           [P1]
#   1b vs 3   → global vs per-layer threshold  (both: ft=10ep)        [P1]
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("1b. UNSTRUCTURED PRUNING  (global L1, WITH fine-tune)")
print("="*65)
model = load_baseline()
prune.global_unstructured(
    get_prunable_layers(model),
    pruning_method=prune.L1Unstructured,
    amount=SPARSITY
)
# Fine-tune with masks active (reparametrization hook enforces zeros automatically)
model = fine_tune(model, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
                  desc="unstructured-ft")
make_permanent(model)
results["1b_unstructured_ft"] = collect_metrics(model, FLOPS_NOTE_MASK)
results["1b_unstructured_ft"]["method_note"] = (
    f"Global L1 unstructured pruning + {FINETUNE_EPOCHS}-epoch fine-tuning. "
    "Paired with 1a (same pruning, no ft) to isolate fine-tuning value. "
    "Paired with 3_magnitude (same ft budget, per-layer threshold) to "
    "isolate global vs per-layer thresholding."
)
model_registry["1b_unstructured_ft"] = model
print(f"  acc={results['1b_unstructured_ft']['accuracy']:.4f}  "
      f"sparsity={results['1b_unstructured_ft']['sparsity_actual']:.3f}")


1a. UNSTRUCTURED PRUNING  (global L1, no fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  acc=0.9320  sparsity=0.499

1b. UNSTRUCTURED PRUNING  (global L1, WITH fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [unstructured-ft] ep   1/10  acc=0.9313
    [unstructured-ft] ep  10/10  acc=0.9330
  acc=0.9330  sparsity=0.499


In [5]:
# ══════════════════════════════════════════════════════════════
# METHOD 2 — STRUCTURED PRUNING (channel removal, Bottleneck-safe)
# ──────────────────────────────────────────────────────────────
# Physically removes CHANNEL_PRUNE_RATIO of internal Bottleneck width
# (conv1 output channels → conv2 input channels) in every block.
# conv3, bn3, and downsample are NEVER touched → residual shapes valid.
# Only method with genuine GFLOPs reduction.
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("2. STRUCTURED PRUNING  (Bottleneck channel removal + fine-tune)")
print("="*65)

def _prune_conv_out(conv, kept_idx):
    n   = len(kept_idx)
    new = nn.Conv2d(conv.in_channels, n, conv.kernel_size, conv.stride,
                    conv.padding, groups=conv.groups,
                    bias=(conv.bias is not None))
    new.weight.data = conv.weight.data[kept_idx].clone()
    if conv.bias is not None:
        new.bias.data = conv.bias.data[kept_idx].clone()
    return new

def _prune_conv_in(conv, kept_idx):
    n   = len(kept_idx)
    new = nn.Conv2d(n, conv.out_channels, conv.kernel_size, conv.stride,
                    conv.padding, groups=conv.groups,
                    bias=(conv.bias is not None))
    new.weight.data = conv.weight.data[:, kept_idx].clone()
    if conv.bias is not None:
        new.bias.data = conv.bias.data.clone()
    return new

def _prune_bn(bn, kept_idx):
    n   = len(kept_idx)
    new = nn.BatchNorm2d(n, eps=bn.eps, momentum=bn.momentum,
                          affine=bn.affine,
                          track_running_stats=bn.track_running_stats)
    if bn.affine:
        new.weight.data = bn.weight.data[kept_idx].clone()
        new.bias.data   = bn.bias.data[kept_idx].clone()
    if bn.track_running_stats:
        new.running_mean        = bn.running_mean[kept_idx].clone()
        new.running_var         = bn.running_var[kept_idx].clone()
        new.num_batches_tracked = bn.num_batches_tracked.clone()
    return new

def prune_resnet50_structured(model, ratio):
    """
    Prune internal Bottleneck width (conv1→conv2) by L2-norm channel scoring.
    conv3, bn3, downsample untouched → residual shape safety guaranteed.
    Shape consistency asserted per block.
    """
    model = copy.deepcopy(model)
    for stage in ['layer1', 'layer2', 'layer3', 'layer4']:
        for block in getattr(model, stage):
            conv1  = block.conv1
            bn1    = block.bn1
            conv2  = block.conv2
            scores = conv1.weight.data.view(conv1.out_channels, -1).norm(p=2, dim=1)
            n_keep = max(1, int(conv1.out_channels * (1 - ratio)))
            kept   = scores.argsort(descending=True)[:n_keep].sort().values
            block.conv1 = _prune_conv_out(conv1, kept)
            block.bn1   = _prune_bn(bn1, kept)
            block.conv2 = _prune_conv_in(conv2, kept)
            assert block.conv1.out_channels == block.conv2.in_channels, \
                f"Shape mismatch after pruning {stage}: " \
                f"conv1.out={block.conv1.out_channels} " \
                f"conv2.in={block.conv2.in_channels}"
    return model

model_struct = prune_resnet50_structured(
    load_baseline(), ratio=CHANNEL_PRUNE_RATIO).to(DEVICE)
model_struct = fine_tune(model_struct, epochs=FINETUNE_EPOCHS,
                          lr=FINETUNE_LR, desc="structured-ft")
results["2_structured"] = collect_metrics(model_struct)
results["2_structured"]["flops_note"] = (
    f"TRUE structured pruning: {CHANNEL_PRUNE_RATIO*100:.0f}% of internal "
    "Bottleneck channels physically removed (conv1+bn1+conv2 per block). "
    "conv3, bn3, downsample untouched — residual shapes guaranteed. "
    "GFLOPs genuinely reduced; measured by thop on rebuilt model."
)
model_registry["2_structured"] = model_struct
print(f"  acc={results['2_structured']['accuracy']:.4f}  "
      f"flops={results['2_structured']['flops_G']:.3f} GFLOPs  "
      f"params={results['2_structured']['params_total']:,}")


2. STRUCTURED PRUNING  (Bottleneck channel removal + fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [structured-ft] ep   1/10  acc=0.9315
    [structured-ft] ep  10/10  acc=0.9313
  acc=0.9313  flops=2.069 GFLOPs  params=18,807,394


In [6]:
# ══════════════════════════════════════════════════════════════
# METHOD 3 — MAGNITUDE PRUNING (per-layer L1 + fine-tune)
# ──────────────────────────────────────────────────────────────
# Each layer independently pruned to SPARSITY by L1 criterion,
# then fine-tuned for FINETUNE_EPOCHS epochs with masks active.
# Reference: Han et al. 2015.
#
# Paired comparisons:
#   3 vs 7   → fine-tune vs no fine-tune   (both: per-layer L1)     [P1]
#   3 vs 4   → L1 criterion vs Taylor      (both: per-layer, ft=10) [P1]
#   3 vs 6   → one-shot vs iterative       (both: total ft=10ep)    [P1]
#   3 vs 1b  → per-layer vs global         (both: ft=10ep)          [P1]
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("3. MAGNITUDE PRUNING  (per-layer L1 + fine-tune)")
print("="*65)
model = load_baseline()
for mod, name in get_prunable_layers(model):
    prune.l1_unstructured(mod, name=name, amount=SPARSITY)
# Fine-tune with masks live — PyTorch hook enforces sparsity on every forward pass
model = fine_tune(model, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
                  desc="magnitude-ft")
make_permanent(model)
results["3_magnitude"] = collect_metrics(model, FLOPS_NOTE_MASK)
results["3_magnitude"]["method_note"] = (
    f"Per-layer L1 magnitude pruning + {FINETUNE_EPOCHS}-epoch fine-tuning. "
    "Han et al. 2015. Central reference method for paired comparisons."
)
model_registry["3_magnitude"] = model
print(f"  acc={results['3_magnitude']['accuracy']:.4f}  "
      f"sparsity={results['3_magnitude']['sparsity_actual']:.3f}")


3. MAGNITUDE PRUNING  (per-layer L1 + fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [magnitude-ft] ep   1/10  acc=0.9319
    [magnitude-ft] ep  10/10  acc=0.9322
  acc=0.9322  sparsity=0.499


In [7]:
# ══════════════════════════════════════════════════════════════
# METHOD 4 — TAYLOR SENSITIVITY PRUNING (per-layer + fine-tune)
# ──────────────────────────────────────────────────────────────
# Importance = (1/N) Σ |∂L/∂w × w| — first-order Taylor expansion of
# loss change from zeroing w (Molchanov et al. 2017).
# NOT movement pruning (Sanh et al. 2020, different algorithm).
#
# (v5→FIX-I) Fine-tuning added matching Method 3's budget (FINETUNE_EPOCHS).
# Previously Method 4 had no fine-tuning, making criterion comparison with
# Method 3 confounded by recovery strategy.
#
# Paired comparison:
#   4 vs 3  → Taylor criterion vs L1 magnitude  (both: ft=FINETUNE_EPOCHS) [P1]
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("4. TAYLOR SENSITIVITY PRUNING  (per-layer |grad×w|, normalized + fine-tune)")
print("="*65)
model = load_baseline()

# Accumulate |grad × weight| over one full epoch
criterion_ts = nn.CrossEntropyLoss()
importance   = {n: torch.zeros_like(p)
                for n, p in model.named_parameters() if p.requires_grad}
model.train()
n_steps = 0
for x, y in train_loader:
    x, y = x.to(DEVICE), y.to(DEVICE)
    model.zero_grad()
    criterion_ts(model(x), y).backward()
    with torch.no_grad():
        for name, p in model.named_parameters():
            if p.requires_grad and p.grad is not None:
                importance[name] += (p.grad * p).abs()
    n_steps += 1

# Normalize by batch count (removes large-tensor / deep-layer bias)
for name in importance:
    importance[name] /= n_steps

# Zero SPARSITY fraction with lowest normalized importance (per-tensor)
model.eval()
with torch.no_grad():
    for name, p in model.named_parameters():
        if name not in importance or importance[name].sum() == 0:
            continue
        scores = importance[name]
        k = max(1, int(scores.numel() * SPARSITY))
        threshold = torch.topk(scores.view(-1), k, largest=False).values.max()
        p.mul_((scores > threshold).float())

# (v5→FIX-I) Fine-tune to match Method 3's recovery budget
model = fine_tune(model, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
                  desc="taylor-ft")
results["4_taylor_sensitivity"] = collect_metrics(model, FLOPS_NOTE_MASK)
results["4_taylor_sensitivity"]["method_note"] = (
    f"Taylor first-order sensitivity (Molchanov et al. 2017). "
    f"Importance = (1/N) Σ |∂L/∂w × w| over N={n_steps} batches, normalized. "
    f"Per-tensor threshold. Fine-tuned {FINETUNE_EPOCHS} epochs (same as Method 3). "
    "NOT movement pruning (Sanh et al. 2020)."
)
model_registry["4_taylor_sensitivity"] = model
print(f"  acc={results['4_taylor_sensitivity']['accuracy']:.4f}  "
      f"sparsity={results['4_taylor_sensitivity']['sparsity_actual']:.3f}")


4. TAYLOR SENSITIVITY PRUNING  (per-layer |grad×w|, normalized + fine-tune)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [taylor-ft] ep   1/10  acc=0.9288
    [taylor-ft] ep  10/10  acc=0.9302
  acc=0.9302  sparsity=0.479


In [8]:
# ══════════════════════════════════════════════════════════════
# METHOD 5 — LOTTERY TICKET HYPOTHESIS
# ──────────────────────────────────────────────────────────────
# Frankle & Carlin, ICLR 2019.
#
# Protocol:
#   θ₀  = random init at SEED (before any gradient update)
#   θ_T = trained from θ₀ for LTH_TRAIN_EPOCHS  (in-script, correct trajectory)
#   m   = global magnitude mask from θ_T  (Conv2d + Linear weights ONLY)
#   Winning ticket = m ⊙ θ₀
#   Retrain with frozen mask for LTH_RETRAIN_EPOCHS
#
# Mask scope (v4→FIX-B, confirmed correct):
#   ONLY Conv2d.weight and Linear.weight are masked.
#   BN affine (γ≈1, β≈0) are small by normalization dynamics, not unimportance.
#   Masking them with a global magnitude threshold kills feature maps silently.
#   BN params are rewound to θ₀ and retrained freely.
#
# Training budget (v5→FIX-III):
#   LTH_TRAIN_EPOCHS = LTH_RETRAIN_EPOCHS = BASELINE_EPOCHS = 100.
#   Matched to baseline training budget for fair ticket quality assessment.
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("5. LOTTERY TICKET HYPOTHESIS  (conv+linear mask, budget-matched)")
print("="*65)

# Step 0: θ₀ — deterministic random init, no training
torch.manual_seed(SEED)
lth_model = build_model().to(DEVICE)
theta_0   = copy.deepcopy(lth_model.state_dict())

# Step 1: θ₀ → θ_T (in-script training — ensures θ₀ and θ_T share trajectory)
print(f"  [LTH] Training θ₀ → θ_T  ({LTH_TRAIN_EPOCHS} epochs) ...")
lth_model = train_model(lth_model, epochs=LTH_TRAIN_EPOCHS,
                         lr=0.1, desc="LTH-train")
theta_T = copy.deepcopy(lth_model.state_dict())

# Step 2: Global magnitude mask from θ_T — Conv2d + Linear weights only
maskable_keys = get_conv_linear_weight_keys(lth_model)
all_vals  = torch.cat([theta_T[k].abs().view(-1) for k in maskable_keys])
threshold = torch.topk(
    all_vals, int(all_vals.numel() * SPARSITY), largest=False
).values.max()
mask = {k: (theta_T[k].abs() > threshold).float().to(DEVICE)
        for k in maskable_keys}
print(f"  [LTH] Mask built over {len(mask)} tensors (conv+linear weights only)")

# Step 3: Winning ticket = m ⊙ θ₀
ticket_state = {}
for k, v in theta_0.items():
    ticket_state[k] = (v.to(DEVICE) * mask[k]) if k in mask else v.to(DEVICE)
ticket_model = build_model().to(DEVICE)
ticket_model.load_state_dict(ticket_state)

# Step 4: Retrain winning ticket with frozen mask
print(f"  [LTH] Retraining winning ticket ({LTH_RETRAIN_EPOCHS} epochs) ...")
ticket_model = train_model(ticket_model, epochs=LTH_RETRAIN_EPOCHS,
                            lr=0.1, desc="LTH-retrain", frozen_mask=mask)

results["5_lottery_ticket"] = collect_metrics(ticket_model, FLOPS_NOTE_MASK)
results["5_lottery_ticket"]["lth_note"] = (
    f"LTH — Frankle & Carlin 2019. "
    f"θ₀ @ SEED={SEED}. "
    f"θ_T: {LTH_TRAIN_EPOCHS}-epoch in-script training (matches baseline budget). "
    f"Global magnitude mask @ {SPARSITY*100:.0f}% on Conv2d+Linear weights only. "
    f"BN affine rewound to θ₀, retrained freely (not masked). "
    f"Ticket retrained {LTH_RETRAIN_EPOCHS} epochs with mask frozen after each step."
)
model_registry["5_lottery_ticket"] = ticket_model
print(f"  acc={results['5_lottery_ticket']['accuracy']:.4f}  "
      f"sparsity={results['5_lottery_ticket']['sparsity_actual']:.3f}")


5. LOTTERY TICKET HYPOTHESIS  (conv+linear mask, budget-matched)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  [LTH] Training θ₀ → θ_T  (100 epochs) ...
    [LTH-train] ep   1/100  acc=0.2099
    [LTH-train] ep  10/100  acc=0.6455
    [LTH-train] ep  20/100  acc=0.7832
    [LTH-train] ep  30/100  acc=0.7923
    [LTH-train] ep  40/100  acc=0.8296
    [LTH-train] ep  50/100  acc=0.8468
    [LTH-train] ep  60/100  acc=0.8699
    [LTH-train] ep  70/100  acc=0.8964
    [LTH-train] ep  80/100  acc=0.9157
    [LTH-train] ep  90/100  acc=0.9404
    [LTH-train] ep 100/100  acc=0.9420
  [LTH] Mask built over 54 tensors (conv+linear weights only)


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  [LTH] Retraining winning ticket (100 epochs) ...
    [LTH-retrain] ep   1/100  acc=0.2291
    [LTH-retrain] ep  10/100  acc=0.6766
    [LTH-retrain] ep  20/100  acc=0.7767
    [LTH-retrain] ep  30/100  acc=0.8122
    [LTH-retrain] ep  40/100  acc=0.7814
    [LTH-retrain] ep  50/100  acc=0.8301


KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════
# METHOD 6 — ITERATIVE PRUNING (N_ROUNDS × prune + fine-tune)
# ──────────────────────────────────────────────────────────────
# (v3→FIX-2) make_permanent() called ONCE after the loop.
# (v5→FIX-II) Total fine-tune epochs = FINETUNE_EPOCHS (same as Methods 3,4).
#   Epochs distributed as ITER_FT_EPOCHS across rounds.
#
# Paired comparison:
#   6 vs 3  → iterative vs one-shot schedule  (both: total ft=FINETUNE_EPOCHS) [P1]
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print(f"6. ITERATIVE PRUNING  ({N_ROUNDS} rounds, total ft={FINETUNE_EPOCHS} epochs)")
print("="*65)
# Compound per-round rate: (1 - S_ROUND)^N_ROUNDS = (1 - SPARSITY)
S_ROUND = 1 - (1 - SPARSITY) ** (1 / N_ROUNDS)
model   = load_baseline()

for r in range(N_ROUNDS):
    ft_ep = ITER_FT_EPOCHS[r]
    prune.global_unstructured(
        get_prunable_layers(model),
        pruning_method=prune.L1Unstructured,
        amount=S_ROUND
    )
    # Fine-tune with masks alive — PyTorch hook enforces zeros each forward pass
    model = fine_tune(model, epochs=ft_ep,
                      lr=FINETUNE_LR * (0.5 ** r),
                      desc=f"iter-r{r+1}(ft={ft_ep}ep)")
    total  = sum(p.numel() for p in model.parameters())
    zeroed = sum(
        (getattr(mod, 'weight_mask') == 0).sum().item()
        for mod in model.modules() if hasattr(mod, 'weight_mask')
    )
    print(f"  Round {r+1}/{N_ROUNDS}  mask-sparsity={zeroed/total:.3f}  "
          f"ft_epochs={ft_ep}")

# (v3→FIX-2) make_permanent outside loop — mask buffers accumulated correctly
make_permanent(model)
nz  = count_nonzero(model)
tot = count_params(model)
print(f"  Final weight sparsity: {1-nz/tot:.3f}  (target={SPARSITY})")
results["6_iterative"] = collect_metrics(model, FLOPS_NOTE_MASK)
results["6_iterative"]["method_note"] = (
    f"{N_ROUNDS}-round global L1 iterative pruning. "
    f"Per-round sparsity: {S_ROUND:.4f} (compounds to {SPARSITY*100:.0f}%). "
    f"Total fine-tune epochs: {sum(ITER_FT_EPOCHS)} = {ITER_FT_EPOCHS} per round. "
    f"Controlled budget: same total ft as Methods 3 and 4 ({FINETUNE_EPOCHS} ep). "
    "make_permanent() called once after all rounds (masks accumulate via AND)."
)
model_registry["6_iterative"] = model
print(f"  acc={results['6_iterative']['accuracy']:.4f}  "
      f"sparsity={results['6_iterative']['sparsity_actual']:.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# METHOD 7 — ONE-SHOT PRUNING (per-layer L1, zero fine-tune)
# ──────────────────────────────────────────────────────────────
# Layer-wise L1, single pass, NO fine-tuning whatsoever.
# "Prune and deploy immediately" baseline.
#
# Paired comparisons:
#   7 vs 1a  → per-layer vs global threshold      (both: no ft)     [P1]
#   7 vs 3   → no ft vs ft=10ep                   (both: per-layer) [P1]
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("7. ONE-SHOT PRUNING  (per-layer L1, zero fine-tune)")
print("="*65)
model = load_baseline()
for mod, name in get_prunable_layers(model):
    prune.l1_unstructured(mod, name=name, amount=SPARSITY)
make_permanent(model)
# Intentionally NO fine-tuning
results["7_one_shot"] = collect_metrics(model, FLOPS_NOTE_MASK)
results["7_one_shot"]["method_note"] = (
    f"Per-layer L1, {SPARSITY*100:.0f}% sparsity, zero fine-tuning. "
    "Paired with 1a (global threshold, no ft) → isolates threshold scope. "
    "Paired with 3 (same criterion, ft=10ep) → isolates fine-tuning value."
)
model_registry["7_one_shot"] = model
print(f"  acc={results['7_one_shot']['accuracy']:.4f}  "
      f"sparsity={results['7_one_shot']['sparsity_actual']:.3f}")

# ══════════════════════════════════════════════════════════════
# END-OF-SCRIPT TIMING PASS (v4→FIX-C)
# All models timed consecutively under identical conditions.
# CUDA synchronization ensures GPU kernels flush before/after timed window.
# ══════════════════════════════════════════════════════════════
TIMING_ORDER = [
    "1a_unstructured_noft",
    "1b_unstructured_ft",
    "2_structured",
    "3_magnitude",
    "4_taylor_sensitivity",
    "5_lottery_ticket",
    "6_iterative",
    "7_one_shot",
]

print("\n" + "="*65)
print(f"TIMING PASS  (warmup={TIMING_WARMUP}, reps={TIMING_REPS}, cuda_sync=True)")
print("="*65)
for key in TIMING_ORDER:
    ms = inference_ms_fn(model_registry[key])
    results[key]["inference_ms"] = ms
    print(f"  {key:<30}  {ms:.3f} ms")

In [ ]:
# ── SAVE JSON ─────────────────────────────────────────────────
output = {
    "_meta": {
        "framework"          : "PFE Pruning Experiments v6",
        "baseline_pth"       : BASELINE,
        "sparsity_target"    : SPARSITY,
        "channel_prune_ratio": CHANNEL_PRUNE_RATIO,
        "device"             : str(DEVICE),
        "seed"               : SEED,
        "baseline_epochs"    : BASELINE_EPOCHS,
        "finetune_epochs"    : FINETUNE_EPOCHS,
        "finetune_lr"        : FINETUNE_LR,
        "iterative_ft_per_round": ITER_FT_EPOCHS,
        "lth_train_epochs"   : LTH_TRAIN_EPOCHS,
        "lth_retrain_epochs" : LTH_RETRAIN_EPOCHS,
        "timing_config": {
            "warmup_reps": TIMING_WARMUP,
            "timed_reps" : TIMING_REPS,
            "cuda_sync"  : True,
            "note"       : (
                "All models timed in a single pass at end of script. "
                "torch.cuda.synchronize() used before/after timed window. "
                "Mask-based methods have identical compute graphs to baseline — "
                "latency differences reflect measurement variance, not real speedup. "
                "Use torch.profiler / NVIDIA Nsight for publication-quality numbers."
            )
        },
        "taxonomy": {
            "mask_based": [
                "1a_unstructured_noft", "1b_unstructured_ft",
                "3_magnitude", "4_taylor_sensitivity",
                "6_iterative", "7_one_shot"
            ],
            "architecture_changing": ["2_structured"],
            "rewind_based"         : ["5_lottery_ticket"],
        },
        "paired_comparisons": {
            "threshold_scope_no_ft" : "1a vs 7   → global vs per-layer threshold (both: no ft)",
            "threshold_scope_with_ft": "1b vs 3   → global vs per-layer threshold (both: ft=10ep)",
            "fine_tune_value"       : "7  vs 3   → no ft vs ft=10ep (both: per-layer L1)",
            "criterion_quality"     : "3  vs 4   → L1 vs Taylor criterion (both: per-layer, ft=10ep)",
            "pruning_schedule"      : "3  vs 6   → one-shot vs iterative (both: total ft=10ep)",
            "ft_value_global"       : "1a vs 1b  → effect of ft on global-pruned model",
        },
        "design_principles": {
            "P1_one_variable": (
                "Each paired comparison isolates exactly one variable. "
                "Fine-tuning budget is controlled across all comparisons."
            ),
            "P2_budget_control": (
                f"All fine-tuning methods use exactly {FINETUNE_EPOCHS} total epochs. "
                f"Iterative splits as {ITER_FT_EPOCHS} per round (sum={sum(ITER_FT_EPOCHS)})."
            ),
            "P3_lth_parity": (
                f"LTH trains for {LTH_TRAIN_EPOCHS}+{LTH_RETRAIN_EPOCHS} epochs. "
                f"Baseline trained for {BASELINE_EPOCHS} epochs. "
                "LTH train/retrain matched to baseline budget."
            ),
            "P4_mask_scope": (
                "LTH masks Conv2d+Linear weights only. "
                "BN affine (γ≈1, β≈0) excluded: small by normalization dynamics, "
                "not by unimportance. Zeroing γ kills feature maps silently."
            ),
            "P5_timing": (
                "All models timed in dedicated end-of-script pass with CUDA sync."
            ),
        },
        "complete_fix_history": {
            "v3_FIX1_1vs7_distinct"     : "Method 7 uses layer-wise (not global) threshold.",
            "v3_FIX2_iterative_sparsity": "make_permanent() outside loop; sparsity reaches target.",
            "v3_FIX3_taylor_naming"     : "Method 4 = Taylor Sensitivity, not Movement Pruning.",
            "v4_FIXA_magnitude_ft"      : "Method 3 now fine-tunes; was identical to Method 7.",
            "v4_FIXB_lth_bn_mask"       : "LTH masks conv+linear only; BN rewound freely.",
            "v4_FIXC_cuda_timing"       : "inference_ms uses torch.cuda.synchronize().",
            "v5_FIXI_taylor_ft"         : "Method 4 now fine-tunes (same budget as Method 3).",
            "v5_FIXII_budget_control"   : f"All ft methods use exactly {FINETUNE_EPOCHS} total epochs.",
            "v5_FIXIII_lth_parity"      : "LTH budget matched to baseline training epochs.",
            "v5_FIXIV_mask_assertion"   : "get_conv_linear_weight_keys() asserts ≥50 keys found.",
            "v5_FIXV_method1b"          : "Added Method 1b (global+ft) for clean threshold comparison.",
        },
        "flops_note": (
            "Mask-based: dense FLOPs unchanged (graph unmodified). "
            "Structured (#2): GFLOPs genuinely reduced, measured on rebuilt model. "
            "Cross-type FLOPs comparison is not apples-to-apples."
        ),
        "structured_pruning_safety": (
            "Only internal Bottleneck width (conv1→conv2) is pruned. "
            "conv3, bn3, downsample untouched — residual shape safety guaranteed."
        ),
    },
    "results": results
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output, f, indent=2)

# ── SUMMARY TABLE ─────────────────────────────────────────────
LABELS = {
    "1a_unstructured_noft" : "1a. Unstructured (global, no-ft)",
    "1b_unstructured_ft"   : "1b. Unstructured (global, +ft)",
    "2_structured"         : "2.  Structured (channel)",
    "3_magnitude"          : "3.  Magnitude (per-layer, +ft)",
    "4_taylor_sensitivity" : "4.  Taylor Sensitivity (+ft)",
    "5_lottery_ticket"     : "5.  LTH",
    "6_iterative"          : "6.  Iterative (3-round, +ft)",
    "7_one_shot"           : "7.  One-Shot (per-layer, no-ft)",
}
TYPES = {
    "1a_unstructured_noft" : "mask",
    "1b_unstructured_ft"   : "mask",
    "2_structured"         : "arch",
    "3_magnitude"          : "mask",
    "4_taylor_sensitivity" : "mask",
    "5_lottery_ticket"     : "rewind",
    "6_iterative"          : "mask",
    "7_one_shot"           : "mask",
}

print("\n" + "="*80)
print(f"ALL DONE — {OUTPUT_JSON}")
print("="*80)
hdr = (f"{'Method':<38} {'Type':<7} {'Acc':>7} {'F1':>7} "
       f"{'Spar':>6} {'MB':>7} {'GFLOPs':>8} {'ms':>8}")
print("\n" + hdr)
print("-" * len(hdr))
for k in TIMING_ORDER:
    m  = results[k]
    sp = m.get('sparsity_actual', 0.0)
    ms = m.get('inference_ms') or 0.0
    print(f"{LABELS.get(k,k):<38} {TYPES.get(k,'?'):<7} "
          f"{m['accuracy']:>7.4f} {m['f1']:>7.4f} {sp:>6.3f} "
          f"{m['size_mb']:>7.2f} {m['flops_G']:>8.3f} {ms:>8.3f}")

print()
print("  Type: mask=weight-masked  arch=physically-rebuilt  rewind=LTH")
print()
print("  Paired comparisons (one variable at a time):")
print("  1a vs 7   → threshold scope    (global vs per-layer,  both no-ft)")
print("  1b vs 3   → threshold scope    (global vs per-layer,  both ft=10ep)")
print("  7  vs 3   → fine-tune value    (no-ft vs ft=10ep,     both per-layer L1)")
print("  3  vs 4   → criterion quality  (L1 vs Taylor,         both ft=10ep)")
print("  3  vs 6   → pruning schedule   (one-shot vs iterative, both ft=10ep total)")
print("  1a vs 1b  → ft value (global)  (no-ft vs ft=10ep,     both global L1)")
print(f"\n  Saved → {OUTPUT_JSON}")

In [ ]:
# ── VERIFICATION BLOCK ────────────────────────────────────────
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)

ok = True

# V1: Methods 3 and 7 must differ (fine-tuning should help)
acc3 = results["3_magnitude"]["accuracy"]
acc7 = results["7_one_shot"]["accuracy"]
v1   = acc3 > acc7
print(f"\nV1  Method 3 (ft) vs Method 7 (no-ft): {acc3:.4f} vs {acc7:.4f}")
print(f"    → {'PASS ✓ (fine-tuning recovered accuracy)' if v1 else 'FAIL — check Method 3 fine-tune'}")
ok = ok and v1

# V2: Methods 3 and 4 criterion comparison is now clean (both have ft)
acc4 = results["4_taylor_sensitivity"]["accuracy"]
print(f"\nV2  Method 3 (L1) vs Method 4 (Taylor) both with ft={FINETUNE_EPOCHS}ep:")
print(f"    acc3={acc3:.4f}  acc4={acc4:.4f}  (result valid regardless of order)")
print(f"    → PASS ✓ (both fine-tuned, comparison is clean)")

# V3: Methods 1b and 3 comparison is clean (both ft=FINETUNE_EPOCHS)
acc1b = results["1b_unstructured_ft"]["accuracy"]
print(f"\nV3  Method 1b (global+ft) vs Method 3 (per-layer+ft):")
print(f"    acc1b={acc1b:.4f}  acc3={acc3:.4f}")
print(f"    → PASS ✓ (both ft={FINETUNE_EPOCHS}ep, isolates threshold scope)")

# V4: Iterative total ft epochs = FINETUNE_EPOCHS
total_iter_ft = sum(ITER_FT_EPOCHS)
v4 = total_iter_ft == FINETUNE_EPOCHS
print(f"\nV4  Iterative total ft epochs: {total_iter_ft}  (target={FINETUNE_EPOCHS})")
print(f"    Per-round: {ITER_FT_EPOCHS}")
print(f"    → {'PASS ✓' if v4 else 'FAIL — ITER_FT_EPOCHS does not sum to FINETUNE_EPOCHS'}")
ok = ok and v4

# V5: Iterative sparsity reached target
sp6 = results["6_iterative"]["sparsity_actual"]
v5  = abs(sp6 - SPARSITY) < 0.05
print(f"\nV5  Iterative sparsity: actual={sp6:.3f}  target={SPARSITY:.2f}")
print(f"    → {'PASS ✓' if v5 else f'FAIL — delta={abs(sp6-SPARSITY):.3f}'}")
ok = ok and v5

# V6: LTH mask covers only conv+linear (no BN keys)
bn_in_mask = [k for k in mask if 'bn' in k or 'downsample.1' in k]
v6 = len(bn_in_mask) == 0
print(f"\nV6  LTH mask keys: {len(mask)} total")
print(f"    BN/downsample-BN keys in mask: {len(bn_in_mask)}")
print(f"    → {'PASS ✓ (conv+linear only)' if v6 else f'FAIL — BN keys present: {bn_in_mask[:3]}'}")
ok = ok and v6

# V7: LTH mask key count sanity
v7 = len(mask) >= 50
print(f"\nV7  LTH maskable key count: {len(mask)}  (expected ≥50 for ResNet-50)")
print(f"    → {'PASS ✓' if v7 else 'FAIL — suspiciously few keys'}")
ok = ok and v7

# V8: Timing stability — no single model >3× another
times = [results[k]["inference_ms"] for k in TIMING_ORDER]
t_min, t_max = min(times), max(times)
ratio = t_max / t_min if t_min > 0 else float('inf')
v8 = ratio <= 3.0
print(f"\nV8  Timing range: {t_min:.2f}–{t_max:.2f} ms  (max/min={ratio:.1f}x)")
print(f"    → {'PASS ✓' if v8 else '⚠ HIGH VARIANCE — re-run on idle GPU'}")
# V8 is advisory, not a hard failure

# V9: Structured pruning genuinely reduces FLOPs
flops_struct   = results["2_structured"]["flops_G"]
flops_baseline = results["1a_unstructured_noft"]["flops_G"]  # same graph as baseline
v9 = flops_struct < flops_baseline
print(f"\nV9  Structured GFLOPs: {flops_struct:.3f}  vs baseline: {flops_baseline:.3f}")
print(f"    → {'PASS ✓ (genuine reduction)' if v9 else 'FAIL — structured should be lower'}")
ok = ok and v9

# V10: No two methods should be bit-for-bit identical (catch accidental duplicates)
import hashlib
def model_hash(key):
    acc = results[key]["accuracy"]
    nz  = results[key]["params_nonzero"]
    return f"{acc:.6f}_{nz}"

hashes = {k: model_hash(k) for k in TIMING_ORDER}
seen   = {}
v10    = True
for k, h in hashes.items():
    if h in seen:
        print(f"\nV10 ⚠ DUPLICATE: {k} and {seen[h]} have identical acc+nonzero")
        v10 = False
    seen[h] = k
if v10:
    print(f"\nV10 No duplicate methods detected ✓")
ok = ok and v10

print("\n" + "="*80)
print(f"OVERALL: {'ALL CHECKS PASSED ✓' if ok else 'SOME CHECKS FAILED — review above'}")
print("="*80)